# GAN-B Baseline: Linux Testing Notebook (Lambda Labs)

Standalone GAN-B testing notebook based on the training notebook style.

- **Input:** healthy RGB images
- **Model outputs:** diseased RGB + lesion mask logits
- **Runs:** folder-wide inference export + validation metrics

Linux/cloud compatible with headless plotting and multiprocess dataloading.

In [ ]:
# 1. Imports
import os
import sys
!{sys.executable} -m pip install tqdm
from tqdm import tqdm
import json
import random
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib
matplotlib.use('Agg')  # headless backend for cloud/SSH
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# 2. Import models and losses
import importlib

# Resolve repo root (notebook is at src/gan/ganB_test_linux.ipynb)
repo_root = Path.cwd().resolve()
if repo_root.name == 'gan':
    repo_root = repo_root.parents[1]
elif repo_root.name == 'src':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f'Repo root: {repo_root}')

import src.models.unet_ganb_dualhead as ganb_unet_mod
import src.models.losses as loss_mod
importlib.reload(ganb_unet_mod)
importlib.reload(loss_mod)

UNetDualHead = ganb_unet_mod.UNetDualHead
PerceptualLoss = loss_mod.PerceptualLoss
SSIMLoss = loss_mod.SSIMLoss
print('GAN-B modules loaded.')

In [ ]:
# 3. Configuration

class Config:
    seed: int = 42
    image_size: int = 512
    batch_size: int = 4
    num_workers: int = 4
    train_split: float = 0.9

    run_name: str = 'ganB_baseline_no_severity_final'
    checkpoint_name: str = 'ganB_baseline_no_severity_final_epoch150.pth'

CFG = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Folders used for validation-metrics pass (same triplet style as training)
HEALTHY_DIR = repo_root / 'data' / 'ganb' / 'healthy'
DISEASED_DIR = repo_root / 'data' / 'ganb' / 'image'
MASK_DIR = repo_root / 'data' / 'ganb' / 'mask'

# Folder used for bulk inference export
INFER_HEALTHY_DIR = repo_root / 'data' / 'ganb' / 'healthy'

CHECKPOINT_DIR = repo_root / 'src' / 'models' / 'checkpoints'
LOG_DIR = repo_root / 'results' / 'logs'
SAMPLE_DIR = repo_root / 'results' / 'figures' / 'training_samples' / 'ganB_baseline_test'
INFER_OUTPUT_IMG_DIR = repo_root / 'results' / 'figures' / 'ganb_inference' / 'generated_diseased'
INFER_OUTPUT_MASK_DIR = repo_root / 'results' / 'figures' / 'ganb_inference' / 'predicted_masks'

for p in [LOG_DIR, SAMPLE_DIR, INFER_OUTPUT_IMG_DIR, INFER_OUTPUT_MASK_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(
    f'Data dirs:\n'
    f'- healthy (val triplets): {HEALTHY_DIR}\n'
    f'- diseased (val triplets): {DISEASED_DIR}\n'
    f'- mask (val triplets): {MASK_DIR}\n'
    f'- inference healthy folder: {INFER_HEALTHY_DIR}'
)

In [ ]:
# 4. Dataset + pairing audit (same style as training notebook)
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def parse_leaf_index_from_image(name: str):
    stem = Path(name).stem
    if stem.startswith('leaf'):
        token = stem[4:]
        if token.isdigit():
            return int(token)
    return None


def parse_leaf_index_from_mask(name: str):
    stem = Path(name).stem
    if stem.startswith('mask_leaf'):
        token = stem[len('mask_leaf'):]
        if token.isdigit():
            return int(token)
    return None


def index_files_by_leaf(files, parser):
    out = {}
    for f in files:
        idx = parser(f)
        if idx is not None:
            out[idx] = f
    return out


def build_triplets_and_audit(healthy_dir: Path, diseased_dir: Path, mask_dir: Path):
    exts = ('.png', '.jpg', '.jpeg')
    healthy_files = sorted([f for f in os.listdir(healthy_dir) if f.lower().endswith(exts)])
    diseased_files = sorted([f for f in os.listdir(diseased_dir) if f.lower().endswith(exts)])
    mask_files = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith(exts)])

    healthy_by_idx = index_files_by_leaf(healthy_files, parse_leaf_index_from_image)
    diseased_by_idx = index_files_by_leaf(diseased_files, parse_leaf_index_from_image)
    masks_by_idx = index_files_by_leaf(mask_files, parse_leaf_index_from_mask)

    common = sorted(set(healthy_by_idx).intersection(diseased_by_idx).intersection(masks_by_idx))
    only_healthy = sorted(set(healthy_by_idx) - set(common))
    only_diseased = sorted(set(diseased_by_idx) - set(common))
    only_masks = sorted(set(masks_by_idx) - set(common))

    triplets = [
        (
            healthy_dir / healthy_by_idx[i],
            diseased_dir / diseased_by_idx[i],
            mask_dir / masks_by_idx[i],
            diseased_by_idx[i],
        )
        for i in common
    ]

    audit = {
        'n_healthy_files': len(healthy_files),
        'n_diseased_files': len(diseased_files),
        'n_mask_files': len(mask_files),
        'n_parsed_healthy': len(healthy_by_idx),
        'n_parsed_diseased': len(diseased_by_idx),
        'n_parsed_masks': len(masks_by_idx),
        'n_triplets': len(triplets),
        'only_healthy_indices': only_healthy[:20],
        'only_diseased_indices': only_diseased[:20],
        'only_mask_indices': only_masks[:20],
    }
    return triplets, audit


def split_triplets(triplets, train_split: float, seed: int):
    indices = list(range(len(triplets)))
    rng = random.Random(seed)
    rng.shuffle(indices)
    n_train = int(len(indices) * train_split)
    train_idx = indices[:n_train]
    val_idx = indices[n_train:]
    train_triplets = [triplets[i] for i in train_idx]
    val_triplets = [triplets[i] for i in val_idx]
    return train_triplets, val_triplets


class DirectGanBDataset(Dataset):
    """Loads pre-generated healthy input + diseased target + lesion mask."""

    def __init__(self, triplets, image_size: int):
        self.triplets = triplets
        self.rgb_tfm = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])
        self.mask_tfm = transforms.Compose([
            transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        healthy_path, diseased_path, mask_path, fname = self.triplets[idx]
        healthy_pil = Image.open(healthy_path).convert('RGB')
        diseased_pil = Image.open(diseased_path).convert('RGB')
        mask_pil = Image.open(mask_path).convert('L')

        mask_np = np.array(mask_pil)
        mask_bin = (mask_np > 127).astype(np.uint8) * 255

        input_t = self.rgb_tfm(healthy_pil)
        target_img_t = self.rgb_tfm(diseased_pil)
        target_mask_t = self.mask_tfm(Image.fromarray(mask_bin)).clamp(0, 1)
        return input_t, target_img_t, target_mask_t, fname


class HealthyOnlyDataset(Dataset):
    """Folder dataset for pure inference on healthy RGB images."""

    def __init__(self, healthy_dir: Path, image_size: int):
        exts = ('.png', '.jpg', '.jpeg')
        self.healthy_dir = healthy_dir
        self.files = sorted([f for f in os.listdir(healthy_dir) if f.lower().endswith(exts)])
        self.rgb_tfm = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = Image.open(self.healthy_dir / fname).convert('RGB')
        inp = self.rgb_tfm(img)
        return inp, fname


set_seed(CFG.seed)
triplets, audit = build_triplets_and_audit(HEALTHY_DIR, DISEASED_DIR, MASK_DIR)
print(json.dumps(audit, indent=2))

audit_path = LOG_DIR / f'{CFG.run_name}_test_dataset_audit.json'
with open(audit_path, 'w', encoding='utf-8') as f:
    json.dump(audit, f, indent=2)
print(f'Audit saved: {audit_path}')

_, val_triplets = split_triplets(triplets, train_split=CFG.train_split, seed=CFG.seed)
print(f'Val triplets: {len(val_triplets)}')

val_ds = DirectGanBDataset(val_triplets, image_size=CFG.image_size)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)

infer_ds = HealthyOnlyDataset(INFER_HEALTHY_DIR, image_size=CFG.image_size)
infer_loader = DataLoader(infer_ds, batch_size=1, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)
print(f'Inference images: {len(infer_ds)}')

In [ ]:
# 5. Build model, losses, and load checkpoint
net_g = UNetDualHead(in_channels=3, image_out_channels=3, mask_out_channels=1).to(DEVICE)
criterion_l1 = nn.L1Loss().to(DEVICE)
criterion_ssim = SSIMLoss().to(DEVICE)
criterion_bce_logits = nn.BCEWithLogitsLoss().to(DEVICE)
_ = PerceptualLoss().to(DEVICE)  # kept for style/compatibility

ckpt_path = CHECKPOINT_DIR / CFG.checkpoint_name
if not ckpt_path.exists():
    raise FileNotFoundError(f'Checkpoint not found: {ckpt_path}')

ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
net_g.load_state_dict(ckpt['netG_state_dict'])
net_g.eval()
print(f'Loaded checkpoint: {ckpt_path}')
print(f"Checkpoint epoch: {ckpt.get('epoch', 'unknown')}")

In [ ]:
# 6. Helpers: dice loss + evaluation
def dice_loss_from_logits(mask_logits, target_mask, eps=1e-6):
    pred = torch.sigmoid(mask_logits)
    pred_flat = pred.view(pred.shape[0], -1)
    target_flat = target_mask.view(target_mask.shape[0], -1)
    inter = (pred_flat * target_flat).sum(dim=1)
    denom = pred_flat.sum(dim=1) + target_flat.sum(dim=1)
    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def evaluate(model_g, loader):
    model_g.eval()
    total_l1 = 0.0
    total_ssim = 0.0
    total_psnr = 0.0
    total_mask_bce = 0.0
    total_mask_dice = 0.0
    n = 0

    with torch.no_grad():
        for inp, target_img, target_mask, _ in tqdm(loader, desc='Evaluating val set'):
            inp = inp.to(DEVICE, non_blocking=True)
            target_img = target_img.to(DEVICE, non_blocking=True)
            target_mask = target_mask.to(DEVICE, non_blocking=True)

            out_img, out_mask_logits = model_g(inp)
            total_l1 += criterion_l1(out_img, target_img).item()
            total_ssim += criterion_ssim.ssim_value(out_img, target_img).item()
            mse_val = torch.mean((out_img - target_img) ** 2).item()
            total_psnr += 10.0 * np.log10(1.0 / (mse_val + 1e-10))
            total_mask_bce += criterion_bce_logits(out_mask_logits, target_mask).item()
            total_mask_dice += dice_loss_from_logits(out_mask_logits, target_mask).item()
            n += 1

    if n == 0:
        return {'l1': 0.0, 'ssim': 0.0, 'psnr': 0.0, 'mask_bce': 0.0, 'mask_dice': 0.0}

    return {
        'l1': total_l1 / n,
        'ssim': total_ssim / n,
        'psnr': total_psnr / n,
        'mask_bce': total_mask_bce / n,
        'mask_dice': total_mask_dice / n,
    }

In [ ]:
# 7. Folder inference export (healthy -> generated diseased + predicted mask)
if len(infer_ds) == 0:
    raise RuntimeError(f'No images found in inference folder: {INFER_HEALTHY_DIR}')

saved = 0
with torch.no_grad():
    for inp, fname in tqdm(infer_loader, desc='Running folder inference'):
        inp = inp.to(DEVICE, non_blocking=True)
        out_img, out_mask_logits = net_g(inp)

        out_img_np = out_img[0].detach().cpu().float().permute(1, 2, 0).numpy().clip(0, 1)
        out_mask_np = torch.sigmoid(out_mask_logits[0, 0]).detach().cpu().float().numpy().clip(0, 1)

        out_img_pil = Image.fromarray((out_img_np * 255).astype(np.uint8))
        out_mask_pil = Image.fromarray((out_mask_np * 255).astype(np.uint8))

        stem = Path(fname[0]).stem
        ext = Path(fname[0]).suffix if Path(fname[0]).suffix else '.png'
        out_img_pil.save(INFER_OUTPUT_IMG_DIR / f'{stem}{ext}')
        out_mask_pil.save(INFER_OUTPUT_MASK_DIR / f'{stem}.png')
        saved += 1

print(f'Inference complete. Saved {saved} images to: {INFER_OUTPUT_IMG_DIR}')
print(f'Inference complete. Saved {saved} masks to: {INFER_OUTPUT_MASK_DIR}')

In [ ]:
# 8. Full validation metrics summary
metrics = evaluate(net_g, val_loader)
print('=' * 50)
print('GAN-B VALIDATION SUMMARY (TEST NOTEBOOK)')
print('=' * 50)
print(f"Val L1:         {metrics['l1']:.6f}")
print(f"Val SSIM:       {metrics['ssim']:.6f}")
print(f"Val PSNR:       {metrics['psnr']:.2f} dB")
print(f"Val Mask BCE:   {metrics['mask_bce']:.6f}")
print(f"Val Mask DiceL: {metrics['mask_dice']:.6f}")
print('=' * 50)

metrics_json = LOG_DIR / f'{CFG.run_name}_test_val_metrics.json'
with open(metrics_json, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
print(f'Metrics saved: {metrics_json}')